# Your first research dataset

## Goal

Imagine you are preparing a short briefing about community facilities. Your question is:
**Which areas had at least three community centres in 2024?**

We will bring a small table into SSDU, select the relevant rows, and save a copy we can
use again later. By the end, you will have a result table and a record of its source.
Allow about 15–20 minutes.

**All places and numbers in this lesson are invented.** They are teaching material,
not official statistics or evidence about actual communities. A six-row table is
small enough to inspect directly; we will use tables throughout this lesson.

## Setup

You can read this whole lesson online, including its results, without installing anything.
To try it yourself, follow [Install and open the tutorial](install.md), then return here.

In JupyterLab, click the first code cell and press **Shift+Enter** to run it and move
on. Run the cells in order. Text below each cell explains what to look for. The first
cell imports our tools and creates a folder called `data/ssdu-tutorial` beside the notebook.
Your example files and saved data will go there.

In [ ]:
from pathlib import Path

import pandas as pd

from ssdu import DatasetQuerySpec, LocalImportSpec, Service, Store

workspace = Path("data/ssdu-tutorial")
workspace.mkdir(parents=True, exist_ok=True)
print("Ready. Tutorial files will be saved in data/ssdu-tutorial.")

## Steps

### 1. Look at the example data

This cell writes our invented CSV: a plain-text table, similar to a spreadsheet.
Each row describes one fictional area in one year. `community_centres` is the number
of centres in that area.

Look for the difference between **0** and **missing**. Meadow Ward has a recorded
zero in 2024, but no reported number in 2023. We cannot treat the missing number as zero.

In [ ]:
source_file = workspace / "community-centres.csv"
source_file.write_text(
    "area,year,community_centres\n"
    "River Ward,2023,4\n"
    "River Ward,2024,5\n"
    "Hill Ward,2023,2\n"
    "Hill Ward,2024,3\n"
    "Meadow Ward,2023,\n"
    "Meadow Ward,2024,0\n",
    encoding="utf-8",
)
print(pd.read_csv(source_file).to_string(index=False, na_rep="missing"))

You should see six rows. `missing` marks the blank entry in the CSV. The numbers
may initially display as `4.0` or `5.0`; we will tell SSDU these are whole-number counts.

### 2. Tell SSDU what the table means

SSDU needs the meaning of the columns as well as the values. We give the table a
title, describe its source, and say what one row represents. The combination of
`area` and `year` identifies a row; together these fields are its **primary key**.

You can reuse this pattern for your own CSV later, changing the descriptions and
column types. Here, `string` means text and `int64` means a whole number.

In [ ]:
details = LocalImportSpec(
    dataset_id="invented-community-centres",
    title="Invented community-centre counts, 2023–2024",
    description="Six independently invented rows for the SSDU tutorial.",
    citation="SSDU teaching example: invented places and counts; not official statistics.",
    licence="MIT — independently invented SSDU teaching material",
    row_grain="One row per fictional area and year.",
    primary_key=["area", "year"],
    columns={
        "area": {"type": "string", "definition": "Fictional area name"},
        "year": {"type": "int64", "definition": "Year described by the count"},
        "community_centres": {
            "type": "int64",
            "definition": "Number of community centres in the fictional area",
            "unit": "centres",
            "missing_values": {"": "No count reported; do not treat as zero"},
        },
    },
)
store = Store(workspace / "saved-data")
dataset = store.import_dataset(source_file, details)
frame = dataset.to_pandas()
print(frame.to_string(index=False, na_rep="missing"))

The saved table still has six rows, including the missing value and the zero.
`<NA>` is how this table displays a missing value; it is not a zero.
SSDU keeps a saved version of the file together with these descriptions. We call
this a **snapshot**. Later edits to your original CSV do not change that snapshot.

### 3. Answer the question

We want rows for **2024** where the number of centres is **at least three**.
The two filters below express those choices. `eq` means “equals”; `gte` means
“greater than or equal to”.

`Service` lets us ask questions of the saved data. The same query interface also
works with supported provider datasets after they have been downloaded.

In [ ]:
service = Service(store.root)
selection = DatasetQuerySpec(
    columns=["area", "year", "community_centres"],
    filters=[
        {"column": "year", "op": "eq", "value": 2024},
        {"column": "community_centres", "op": "gte", "value": 3},
    ],
    sort=[{"column": "area"}],
)
result = service.query_dataset(dataset.snapshot_id, selection)
print(pd.DataFrame(result["rows"]).to_string(index=False))

You should see **Hill Ward (3)** and **River Ward (5)**, both for 2024. Meadow Ward
has zero centres in 2024, so it does not meet our filter.

This answers the question for our invented table. Counts alone would not tell us
whether an area has enough facilities: that would also need population, capacity,
accessibility and other evidence.

### 4. Keep the source with your result

SSDU saved the citation and the meaning of the count. Here we print only the fields
we need, rather than the complete technical description.

In [ ]:
description = dataset.contract
print("Title:", description["title"])
print("Source:", description["source"]["citation"])
print("One row:", description["row_grain"])
print("Unit:", details.columns["community_centres"].unit)

For your own data, give the original publisher, dataset and release details in
the citation. Keep units, missing-value explanations and any restrictions on use.
SSDU can preserve those details; you still need to check that they are correct.

### 5. Reopen the same saved version

A snapshot has a unique identifier, like a reference number. Save it in a small text
file so you can reopen this exact version later. Keep the `saved-data` folder too:
the identifier points to that saved data, rather than containing the data itself.

In [ ]:
snapshot_note = workspace / "snapshot-id.txt"
snapshot_note.write_text(dataset.snapshot_id, encoding="utf-8")

saved_id = snapshot_note.read_text(encoding="utf-8").strip()
reopened_store = Store(workspace / "saved-data")
reopened = reopened_store.open_dataset(saved_id)
print(reopened.to_pandas().to_string(index=False, na_rep="missing"))

The same six rows should appear. Reopening this snapshot reads SSDU's saved copy;
it does not download anything or reread the original CSV. In a later session,
import `Store` and `Path`, set `workspace` as in Setup, and run this cell from
`saved_id = ...` onwards.

## Checks

This final cell checks the lesson's result, the missing value, the recorded zero,
and the reopened table. A successful run prints the message below.

In [ ]:
expected_rows = [
    {"area": "Hill Ward", "year": 2024, "community_centres": 3},
    {"area": "River Ward", "year": 2024, "community_centres": 5},
]
assert result["rows"] == expected_rows
assert len(frame) == 6
assert frame["community_centres"].isna().sum() == 1
assert (frame["community_centres"] == 0).sum() == 1
pd.testing.assert_frame_equal(frame, reopened.to_pandas())
assert reopened.contract == dataset.contract
print("All checks passed: your result and saved data match the lesson.")

## Next Steps

Try changing the year filter from `2024` to `2023` in Step 3, then run that cell.
Only **River Ward (4)** meets the threshold. Put it back to `2024` before rerunning
the final checks, which expect the original question.

When you are ready:

- [Understand the data sources SSDU supports](../reference/sources.md).
- [Inspect and query your own CSV or Parquet data](../guides/dataset-api.md).
- [Learn how provider data use the same interface](../guides/dataset-examples.md).
- [Transfer saved data to another machine](quickstart.md).

The beginner lesson uses local data. Provider downloads also need a dataset
identifier and provider-specific choices, such as geography and dates. A common
interface preserves those differences; it does not make every statistic comparable.